# Phase 3: Text-Only Model Robustness Experiments - Fashion Classification

## Objective
Test **text-only model** robustness across **30 different data splits** to evaluate stability and generalization. Each experiment uses a different random seed for data splitting, creating unique train/val/test distributions.

## Strategy
- Use **same 30 random seeds** as multi-modal experiments (for fair comparison)
- **Same configuration** as multi-modal: LR=5e-5, BS=32, ES=5, Dropout=0.5, WD=1e-4
- Each seed creates different train/val/test split
- Model initialization: Fixed seed (42) for reproducibility
- **Text-only architecture**: FashionBERT → Classifier (no visual features)
- Compute per-class metrics during training (at best epoch)
- Track metrics across all experiments
- Statistical analysis (mean ± std, CI, etc.)

## Model Architecture
- **Text-Only**: FashionBERT (768-dim) → Classifier (768 → 256 → 128 → 14)
- **Multi-Modal** (for comparison): CLIP (512-dim) + FashionBERT (768-dim) → Fusion (512-dim) → Classifier (512 → 256 → 128 → 14)

## Output
- 30 JSON result files (one per seed) with per-class metrics included
- 30 model checkpoints
- 30 learning curves plots
- Statistical analysis and summary tables (overall and per-class)
- Comprehensive reports and visualizations


## 1. Configuration and Setup


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import json
import os

# Repository root (chdir so relative paths work from notebooks/)
_walk = os.path.abspath(os.getcwd())
for _ in range(10):
    if os.path.isdir(os.path.join(_walk, 'experiments')) and os.path.isdir(os.path.join(_walk, 'data')):
        PROJECT_ROOT = _walk
        break
    _walk = os.path.dirname(_walk)
else:
    PROJECT_ROOT = os.path.abspath(os.getcwd())
os.chdir(PROJECT_ROOT)
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
import warnings
import time
from datetime import datetime
from scipy import stats

# Suppress tokenizers parallelism warnings
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings('ignore')

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ============================================
# FIXED CONFIGURATION (Same as Multi-Modal for Fair Comparison)
# ============================================
LEARNING_RATE = 5e-5
BATCH_SIZE = 32
EARLY_STOPPING_PATIENCE = 5
DROPOUT = 0.5
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 20

# Fixed seed for model initialization (reproducibility)
MODEL_INIT_SEED = 42

# Data split ratios (same as multi-modal)
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15

# ============================================
# LOAD SEEDS FROM ROOT FOLDER
# ============================================
seeds_file = os.path.join(PROJECT_ROOT, 'data', 'processed', 'seeds_list.txt')
if os.path.exists(seeds_file):
    with open(seeds_file, 'r') as f:
        content = f.read()
    # Extract seed values from file (handle various formats)
    SEEDS = []
    import re
    # Try to find all integers in the file
    numbers = re.findall(r'\d+', content)
    for num_str in numbers:
        try:
            seed_value = int(num_str)
            if 1 <= seed_value <= 500:  # Valid seed range
                SEEDS.append(seed_value)
        except ValueError:
            continue
    # Remove duplicates and sort
    SEEDS = sorted(list(set(SEEDS)))
    print(f"✅ Loaded {len(SEEDS)} seeds from {seeds_file}")
    print(f"   Seeds: {SEEDS}")
    if len(SEEDS) != 30:
        print(f"⚠️  Warning: Expected 30 seeds, found {len(SEEDS)}")
else:
    print(f"⚠️  Seeds file not found: {seeds_file}")
    print("   Please ensure seeds_list.txt exists in the root folder.")
    SEEDS = []

# Output directories
EXPERIMENT_ROOT = "experiments/textonly_robustness"
METRICS_DIR = os.path.join(EXPERIMENT_ROOT, "metrics")
ARTIFACTS_DIR = os.path.join(EXPERIMENT_ROOT, "artifacts")
os.makedirs(METRICS_DIR, exist_ok=True)
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
os.makedirs(os.path.join(METRICS_DIR, "experiments"), exist_ok=True)
os.makedirs(os.path.join(ARTIFACTS_DIR, "models"), exist_ok=True)
os.makedirs(os.path.join(ARTIFACTS_DIR, "learning_curves"), exist_ok=True)
os.makedirs(os.path.join(ARTIFACTS_DIR, "comparison_plots"), exist_ok=True)
os.makedirs(os.path.join(ARTIFACTS_DIR, "per_class_visualizations"), exist_ok=True)

print(f"\n✅ Configuration loaded:")
print(f"   Learning Rate: {LEARNING_RATE}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Early Stopping Patience: {EARLY_STOPPING_PATIENCE}")
print(f"   Dropout: {DROPOUT}")
print(f"   Weight Decay: {WEIGHT_DECAY}")
print(f"   Max Epochs: {MAX_EPOCHS}")
print(f"\n💾 Experiment: {EXPERIMENT_ROOT}\n   metrics: {METRICS_DIR}\n   artifacts: {ARTIFACTS_DIR}")
print(f"   Model init seed: {MODEL_INIT_SEED} (fixed for all experiments)")


## 2. Load Full Dataset


In [ ]:
# Load caption dataset
print("Loading caption dataset...")
df = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'processed', 'caption_dataset_final_full.csv'))

# Filter for successful captions
df_success = df[df['status'] == 'success'].copy()
print(f"Total images with captions: {len(df_success)}")

# Extract style categories
df_success['style'] = df_success['style'].str.strip()  # Clean whitespace
all_styles = sorted(df_success['style'].unique())
style_to_idx = {style: idx for idx, style in enumerate(all_styles)}
idx_to_style = {idx: style for style, idx in style_to_idx.items()}
num_classes = len(all_styles)

print(f"\nStyle categories: {num_classes}")
print(f"Styles: {all_styles}")

# Use full dataset (100%)
print("\nUsing full dataset (100%)")
df_full = df_success.copy()
print(f"Full dataset size: {len(df_full)}")

# Create caption dictionary
captions_dict = {}
for _, row in df_full.iterrows():
    captions_dict[row['image_path']] = row['caption']

print(f"\nCaption dictionary created: {len(captions_dict)} entries")
print("\n⚠️  Note: Data splits will be created per experiment with different random seeds")


## 3. Load Pre-trained Models


In [ ]:
# Load FashionBERT (Text-Only Model)
from transformers import AutoTokenizer, AutoModel
print("Loading FashionBERT...")
fashionbert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
fashionbert_model = AutoModel.from_pretrained('bert-base-uncased').to(device)
fashionbert_model.eval()
print("FashionBERT model loaded!")


## 4. Dataset and Model Classes


In [ ]:
class FashionTextOnlyDataset(Dataset):
    """Text-only dataset for fashion style classification"""
    
    def __init__(self, df, captions_dict, style_to_idx):
        self.df = df.reset_index(drop=True)
        self.captions_dict = captions_dict
        self.style_to_idx = style_to_idx
        
        # Filter out entries without captions
        self.valid_indices = []
        for idx in range(len(self.df)):
            row = self.df.iloc[idx]
            image_path = row['image_path']
            if image_path in captions_dict:
                self.valid_indices.append(idx)
        
        print(f"Dataset initialized with {len(self.valid_indices)} valid samples (out of {len(self.df)})")
    
    def __len__(self):
        return len(self.valid_indices)
    
    def __getitem__(self, idx):
        actual_idx = self.valid_indices[idx]
        row = self.df.iloc[actual_idx]
        
        # Get caption
        image_path = row['image_path']
        caption = self.captions_dict.get(image_path, "No caption available")
        
        # Get label
        style = row['style']
        label = self.style_to_idx[style]
        
        return {
            'caption': caption,
            'label': label,
            'style': style,
            'image_path': image_path
        }

class TextOnlyFashionClassifier(nn.Module):
    """Text-only fashion style classifier using FashionBERT"""
    
    def __init__(self, fashionbert_model, fashionbert_tokenizer, num_classes, dropout=0.5, textual_dim=768, device=None):
        super(TextOnlyFashionClassifier, self).__init__()
        
        self.fashionbert_model = fashionbert_model
        self.fashionbert_tokenizer = fashionbert_tokenizer
        self.device = device if device is not None else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.textual_dim = textual_dim
        
        # Classification head (same structure as multi-modal: 256 → 128 → num_classes)
        # Input dimension is 768 (BERT output) instead of 512 (fusion output)
        self.classifier = nn.Sequential(
            nn.Linear(768, 256),  # Changed from 512 to 768
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, captions):
        # Extract textual features using FashionBERT
        with torch.no_grad():
            inputs = self.fashionbert_tokenizer(
                captions, 
                return_tensors='pt', 
                padding=True, 
                truncation=True, 
                max_length=128
            ).to(self.device)
            outputs = self.fashionbert_model(**inputs)
            textual_features = outputs.last_hidden_state[:, 0, :]  # [CLS] token
        
        # Classification
        logits = self.classifier(textual_features)
        
        return logits

print("✅ Model classes defined!")


## 5. Training and Evaluation Functions


In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch in tqdm(train_loader, desc="Training", leave=False):
        captions = batch['caption']
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        logits = model(captions)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(logits.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    return total_loss / len(train_loader), 100. * correct / total

def validate_epoch(model, val_loader, criterion, device):
    """Validate for one epoch - returns predictions and labels for per-class metrics"""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation", leave=False):
            captions = batch['caption']
            labels = batch['label'].to(device)
            
            logits = model(captions)
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            _, predicted = torch.max(logits.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Calculate macro F1
    val_macro_f1 = f1_score(all_labels, all_predictions, average='macro', zero_division=0) if len(all_predictions) > 0 else 0.0
    
    return total_loss / len(val_loader), 100. * correct / total, all_predictions, all_labels, val_macro_f1

def evaluate_with_per_class_metrics(model, data_loader, criterion, device, all_styles):
    """
    Evaluate model and return all metrics including per-class metrics
    
    Returns:
        loss, accuracy, predictions, labels, macro_f1, macro_precision, macro_recall,
        per_class_f1, per_class_precision, per_class_recall, per_class_accuracy
    """
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Evaluating", leave=False):
            captions = batch['caption']
            labels = batch['label'].to(device)
            
            logits = model(captions)
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            _, predicted = torch.max(logits.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Calculate overall metrics
    accuracy = 100.0 * correct / total if total > 0 else 0.0
    avg_loss = total_loss / len(data_loader) if len(data_loader) > 0 else 0.0
    
    # Macro-averaged metrics
    macro_f1 = f1_score(all_labels, all_predictions, average='macro', zero_division=0)
    macro_precision = precision_score(all_labels, all_predictions, average='macro', zero_division=0)
    macro_recall = recall_score(all_labels, all_predictions, average='macro', zero_division=0)
    
    # Per-class metrics
    per_class_f1 = f1_score(all_labels, all_predictions, average=None, zero_division=0)
    per_class_precision = precision_score(all_labels, all_predictions, average=None, zero_division=0)
    per_class_recall = recall_score(all_labels, all_predictions, average=None, zero_division=0)
    
    # Per-class accuracy (for each class: TP / (TP + FN) = recall, but we compute explicitly)
    per_class_accuracy = []
    for class_idx in range(len(all_styles)):
        class_mask = np.array(all_labels) == class_idx
        if np.sum(class_mask) > 0:
            class_correct = np.sum((np.array(all_predictions) == class_idx) & class_mask)
            class_acc = class_correct / np.sum(class_mask)
        else:
            class_acc = 0.0
        per_class_accuracy.append(class_acc)
    
    # Convert to dictionaries with style names
    per_class_f1_dict = {all_styles[i]: float(per_class_f1[i]) for i in range(len(all_styles))}
    per_class_precision_dict = {all_styles[i]: float(per_class_precision[i]) for i in range(len(all_styles))}
    per_class_recall_dict = {all_styles[i]: float(per_class_recall[i]) for i in range(len(all_styles))}
    per_class_accuracy_dict = {all_styles[i]: float(per_class_accuracy[i]) for i in range(len(all_styles))}
    
    return (
        avg_loss, accuracy, all_predictions, all_labels,
        macro_f1, macro_precision, macro_recall,
        per_class_f1_dict, per_class_precision_dict, per_class_recall_dict, per_class_accuracy_dict
    )

print("✅ Training and evaluation functions defined!")


In [ ]:
def run_textonly_experiment(seed_value, seed_idx, df_full, captions_dict, style_to_idx,
                            fashionbert_model, fashionbert_tokenizer,
                            num_classes, device, all_styles):
    """
    Run a single text-only robustness experiment with given seed
    
    Args:
        seed_value: Random seed value for data splitting
        seed_idx: Index of seed in SEEDS list
        df_full: Full dataset DataFrame
        captions_dict: Dictionary mapping image paths to captions
        style_to_idx: Dictionary mapping style names to indices
        fashionbert_model: Pre-trained BERT model
        fashionbert_tokenizer: BERT tokenizer
        num_classes: Number of classes
        device: Device
        all_styles: List of style names
    
    Returns:
        Dictionary with all results and metrics (including per-class metrics)
    """
    
    print(f"\n{'='*70}")
    print(f"Experiment {seed_idx}/{len(SEEDS)}: Seed {seed_value}")
    print(f"{'='*70}")
    
    # Check if result already exists (resume capability)
    result_file = os.path.join(METRICS_DIR, "experiments", f"seed_{seed_value}_results.json")
    if os.path.exists(result_file):
        print(f"  ⏭️  Result already exists, skipping...")
        with open(result_file, 'r') as f:
            return json.load(f)
    
    # Set fixed seed for model initialization (same for all experiments)
    torch.manual_seed(MODEL_INIT_SEED)
    np.random.seed(MODEL_INIT_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(MODEL_INIT_SEED)
    
    # Create stratified train/val/test splits with this seed
    print(f"  Creating data splits with random_state={seed_value}...")
    train_df, temp_df = train_test_split(
        df_full,
        test_size=(VAL_RATIO + TEST_RATIO),
        stratify=df_full['style'],
        random_state=seed_value  # Different seed for each experiment
    )
    
    val_df, test_df = train_test_split(
        temp_df,
        test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO),
        stratify=temp_df['style'],
        random_state=seed_value  # Same seed
    )
    
    print(f"  Split sizes: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")
    
    # Create datasets (text-only, no transforms needed)
    train_dataset = FashionTextOnlyDataset(train_df, captions_dict, style_to_idx)
    val_dataset = FashionTextOnlyDataset(val_df, captions_dict, style_to_idx)
    test_dataset = FashionTextOnlyDataset(test_df, captions_dict, style_to_idx)
    
    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    # Compute class weights
    class_weights = compute_class_weight(
        'balanced',
        classes=np.array(list(style_to_idx.values())),
        y=train_df['style'].map(style_to_idx).values
    )
    class_weights = torch.FloatTensor(class_weights).to(device)
    
    # Initialize model (with fixed seed)
    model = TextOnlyFashionClassifier(
        fashionbert_model=fashionbert_model,
        fashionbert_tokenizer=fashionbert_tokenizer,
        num_classes=num_classes,
        dropout=DROPOUT,
        device=device
    ).to(device)
    
    # Setup training
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(
        model.parameters(), 
        lr=LEARNING_RATE, 
        weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)
    
    # Training tracking
    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []
    val_macro_f1s = []
    learning_rates = []
    
    best_val_macro_f1 = 0
    best_val_loss = float('inf')
    best_epoch = 0
    patience_counter = 0
    early_stopped = False
    
    start_time = time.time()
    
    # Training loop with early stopping
    for epoch in range(MAX_EPOCHS):
        # Train
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        
        # Validate
        val_loss, val_acc, val_preds, val_labels, val_macro_f1 = validate_epoch(
            model, val_loader, criterion, device
        )
        
        # Update scheduler
        scheduler.step()
        learning_rates.append(scheduler.get_last_lr()[0])
        
        # Store metrics
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)
        val_macro_f1s.append(val_macro_f1)
        
        # Track best Macro F1 (for saving & early stopping)
        if val_macro_f1 > best_val_macro_f1:
            best_val_macro_f1 = val_macro_f1
            best_epoch = epoch + 1
            patience_counter = 0
            # Save best model
            torch.save(model.state_dict(), 
                      os.path.join(ARTIFACTS_DIR, "models", f"seed_{seed_value}_best_model.pth"))
        else:
            patience_counter += 1
        
        # Track best loss (for overfitting detection)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
        
        # Early stopping (based on Macro F1)
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            early_stopped = True
            print(f"  Early stopping at epoch {epoch+1} (no improvement for {EARLY_STOPPING_PATIENCE} epochs)")
            break
        
        # Print progress
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1}/{MAX_EPOCHS}: "
                  f"Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}, "
                  f"Val Macro F1={val_macro_f1:.4f}")
    
    total_time = time.time() - start_time
    
    # Load best model for final evaluation
    model.load_state_dict(torch.load(
        os.path.join(ARTIFACTS_DIR, "models", f"seed_{seed_value}_best_model.pth")))
    model.eval()
    
    # Re-evaluate validation set at best epoch with per-class metrics
    print(f"  Computing per-class metrics on validation set (best epoch)...")
    (val_loss, val_acc, val_preds, val_labels,
     val_macro_f1, val_macro_precision, val_macro_recall,
     val_per_class_f1, val_per_class_precision, val_per_class_recall, val_per_class_accuracy) = evaluate_with_per_class_metrics(
        model, val_loader, criterion, device, all_styles
    )
    
    # Final evaluation on test set with per-class metrics
    print(f"  Computing per-class metrics on test set...")
    (test_loss, test_acc, test_preds, test_labels,
     test_macro_f1, test_macro_precision, test_macro_recall,
     test_per_class_f1, test_per_class_precision, test_per_class_recall, test_per_class_accuracy) = evaluate_with_per_class_metrics(
        model, test_loader, criterion, device, all_styles
    )
    
    # Detect overfitting
    if len(val_losses) > best_epoch:
        val_loss_after_best = min(val_losses[best_epoch:])
        overfitting_detected = val_loss_after_best > best_val_loss * 1.05
    else:
        overfitting_detected = False
    
    # Calculate train/val gap at best epoch
    train_val_gap = train_losses[best_epoch - 1] - best_val_loss if best_epoch > 0 else 0
    
    # Create learning curves plot
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Loss curves
    axes[0, 0].plot(train_losses, label='Train Loss', color='blue', linewidth=2)
    axes[0, 0].plot(val_losses, label='Val Loss', color='red', linewidth=2)
    axes[0, 0].axvline(x=best_epoch-1, color='green', linestyle='--', alpha=0.7, label=f'Best Epoch {best_epoch}')
    axes[0, 0].set_title('Training and Validation Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Accuracy curves
    axes[0, 1].plot(train_accs, label='Train Acc', color='blue', linewidth=2)
    axes[0, 1].plot(val_accs, label='Val Acc', color='red', linewidth=2)
    axes[0, 1].axvline(x=best_epoch-1, color='green', linestyle='--', alpha=0.7)
    axes[0, 1].set_title('Training and Validation Accuracy')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy (%)')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Macro F1 curve
    axes[1, 0].plot(val_macro_f1s, label='Val Macro F1', color='green', linewidth=2)
    axes[1, 0].axvline(x=best_epoch-1, color='red', linestyle='--', alpha=0.7)
    axes[1, 0].axhline(y=best_val_macro_f1, color='red', linestyle='--', alpha=0.7, 
                       label=f'Best: {best_val_macro_f1:.4f}')
    axes[1, 0].set_title('Validation Macro F1-Score')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Macro F1')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Summary text
    axes[1, 1].axis('off')
    summary_text = f"""
Seed: {seed_value} (Experiment {seed_idx}/{len(SEEDS)})
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Best Epoch: {best_epoch}
Early Stopped: {early_stopped}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Best Val Macro F1: {best_val_macro_f1:.4f}
Test Macro F1: {test_macro_f1:.4f}
Test Accuracy: {test_acc:.2f}%
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Overfitting: {'Yes' if overfitting_detected else 'No'}
Training Time: {total_time/60:.1f} minutes
    """
    axes[1, 1].text(0.1, 0.5, summary_text, fontsize=10, family='monospace',
                    verticalalignment='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.suptitle(f'Learning Curves: Seed {seed_value} (Text-Only Model)', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    # Save plot
    plot_path = os.path.join(ARTIFACTS_DIR, "learning_curves", f"seed_{seed_value}_learning_curves.png")
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    # Prepare results dictionary (with per-class metrics included)
    results = {
        "experiment_id": f"seed_{seed_value}",
        "seed_value": seed_value,
        "seed_index": seed_idx,
        "timestamp": datetime.now().isoformat(),
        "model_type": "text_only",
        "configuration": {
            "learning_rate": float(LEARNING_RATE),
            "batch_size": BATCH_SIZE,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "dropout": DROPOUT,
            "weight_decay": float(WEIGHT_DECAY),
            "max_epochs": MAX_EPOCHS,
            "model_init_seed": MODEL_INIT_SEED,
            "data_split_seed": seed_value,
            "dataset_size": "100% (full dataset)"
        },
        "training_info": {
            "total_epochs": len(train_losses),
            "best_epoch": best_epoch,
            "early_stopped": early_stopped,
            "total_time_minutes": float(total_time / 60)
        },
        "validation_metrics": {
            "best_val_macro_f1": float(val_macro_f1),
            "best_val_accuracy": float(val_acc),
            "best_val_macro_precision": float(val_macro_precision),
            "best_val_macro_recall": float(val_macro_recall),
            "best_val_loss": float(best_val_loss),
            "final_val_macro_f1": float(val_macro_f1s[-1]),
            "final_val_accuracy": float(val_accs[-1]),
            "final_val_loss": float(val_losses[-1]),
            "per_class_metrics": {
                "f1": val_per_class_f1,
                "precision": val_per_class_precision,
                "recall": val_per_class_recall,
                "accuracy": val_per_class_accuracy
            }
        },
        "test_metrics": {
            "test_macro_f1": float(test_macro_f1),
            "test_accuracy": float(test_acc),
            "test_macro_precision": float(test_macro_precision),
            "test_macro_recall": float(test_macro_recall),
            "test_loss": float(test_loss),
            "per_class_metrics": {
                "f1": test_per_class_f1,
                "precision": test_per_class_precision,
                "recall": test_per_class_recall,
                "accuracy": test_per_class_accuracy
            }
        },
        "overfitting_analysis": {
            "overfitting_detected": overfitting_detected,
            "best_val_loss": float(best_val_loss),
            "val_loss_after_best": float(val_losses[best_epoch:][0]) if len(val_losses) > best_epoch else float(val_losses[-1]),
            "increase_percentage": float((val_losses[best_epoch:][0] - best_val_loss) / best_val_loss * 100) if len(val_losses) > best_epoch else 0.0,
            "train_val_gap": float(train_val_gap)
        },
        "training_curves": {
            "train_losses": [float(x) for x in train_losses],
            "val_losses": [float(x) for x in val_losses],
            "train_accs": [float(x) for x in train_accs],
            "val_accs": [float(x) for x in val_accs],
            "val_macro_f1s": [float(x) for x in val_macro_f1s],
            "learning_rates": [float(x) for x in learning_rates]
        },
        "data_split_info": {
            "train_size": len(train_df),
            "val_size": len(val_df),
            "test_size": len(test_df)
        }
    }
    
    # Save results JSON
    json_path = os.path.join(METRICS_DIR, "experiments", f"seed_{seed_value}_results.json")
    with open(json_path, 'w') as f:
        json.dump(results, f, indent=2)
    
    print(f"  ✅ Completed: Best Val Macro F1={best_val_macro_f1:.4f}, "
          f"Test Macro F1={test_macro_f1:.4f}, "
          f"Overfitting={'Yes' if overfitting_detected else 'No'}")
    
    return results

print("✅ Experiment runner function defined!")


In [ ]:
# Run all text-only robustness experiments
all_results = []
failed_seeds = []

print(f"\n{'='*70}")
print(f"STARTING TEXT-ONLY ROBUSTNESS EXPERIMENTS")
print(f"Total experiments: {len(SEEDS)}")
print(f"Dataset: Full (100%) - {len(df_full)} images")
print(f"Estimated time: ~{len(SEEDS) * 1.5:.1f} hours (faster than multi-modal)")
print(f"{'='*70}")

for seed_idx, seed_value in enumerate(SEEDS, 1):
    try:
        result = run_textonly_experiment(
            seed_value=seed_value,
            seed_idx=seed_idx,
            df_full=df_full,
            captions_dict=captions_dict,
            style_to_idx=style_to_idx,
            fashionbert_model=fashionbert_model,
            fashionbert_tokenizer=fashionbert_tokenizer,
            num_classes=num_classes,
            device=device,
            all_styles=all_styles
        )
        
        all_results.append(result)
        
        # Progress update
        remaining = len(SEEDS) - seed_idx
        print(f"\n✅ Progress: {seed_idx}/{len(SEEDS)} completed, {remaining} remaining")
        
    except Exception as e:
        print(f"\n❌ Error in seed {seed_value}: {e}")
        failed_seeds.append((seed_value, str(e)))
        import traceback
        traceback.print_exc()
        continue

print(f"\n{'='*70}")
print(f"ALL EXPERIMENTS COMPLETED!")
print(f"  Successful: {len(all_results)}/{len(SEEDS)}")
if failed_seeds:
    print(f"  Failed: {len(failed_seeds)} seeds")
    print(f"  Failed seeds: {[s[0] for s in failed_seeds]}")
print(f"{'='*70}")

# Save summary of completed experiments
summary = {
    "total_seeds": len(SEEDS),
    "successful_experiments": len(all_results),
    "failed_experiments": len(failed_seeds),
    "failed_seeds": [{"seed": s[0], "error": s[1]} for s in failed_seeds],
    "completed_seeds": [r["seed_value"] for r in all_results]
}

with open(os.path.join(METRICS_DIR, "experiments_summary.json"), 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n✅ Experiments summary saved to: {os.path.join(METRICS_DIR, 'experiments_summary.json')}")


## 8. Load All Results and Compute Statistics


In [ ]:
# Load all results if not already loaded
if len(all_results) == 0:
    print("Loading results from JSON files...")
    all_results = []
    for seed_value in SEEDS:
        result_file = os.path.join(METRICS_DIR, "experiments", f"seed_{seed_value}_results.json")
        if os.path.exists(result_file):
            with open(result_file, 'r') as f:
                all_results.append(json.load(f))
    print(f"Loaded {len(all_results)} results")

if len(all_results) == 0:
    print("⚠️  No results found! Please run experiments first.")
else:
    print(f"✅ Loaded {len(all_results)} results with per-class metrics")


## 9. Statistical Analysis: Overall Metrics


In [ ]:
def calculate_stats(values, name):
    """Calculate comprehensive statistics for a metric"""
    mean_val = np.mean(values)
    std_val = np.std(values)
    min_val = np.min(values)
    max_val = np.max(values)
    median_val = np.median(values)
    q25 = np.percentile(values, 25)
    q75 = np.percentile(values, 75)
    cv = (std_val / mean_val * 100) if mean_val != 0 else 0  # Coefficient of Variation
    
    # 95% Confidence Interval
    if len(values) > 1:
        ci = stats.t.interval(0.95, len(values)-1, loc=mean_val, scale=stats.sem(values))
        ci_lower, ci_upper = ci
    else:
        ci_lower, ci_upper = mean_val, mean_val
    
    return {
        "metric": name,
        "mean": float(mean_val),
        "std": float(std_val),
        "min": float(min_val),
        "max": float(max_val),
        "median": float(median_val),
        "q25": float(q25),
        "q75": float(q75),
        "cv_percent": float(cv),
        "ci_95_lower": float(ci_lower),
        "ci_95_upper": float(ci_upper),
        "n": len(values)
    }

if len(all_results) > 0:
    # Extract overall metrics
    test_f1s = [r['test_metrics']['test_macro_f1'] for r in all_results]
    test_accs = [r['test_metrics']['test_accuracy'] for r in all_results]
    test_precisions = [r['test_metrics'].get('test_macro_precision', 0) for r in all_results]
    test_recalls = [r['test_metrics'].get('test_macro_recall', 0) for r in all_results]
    
    best_val_f1s = [r['validation_metrics']['best_val_macro_f1'] for r in all_results]
    best_val_accs = [r['validation_metrics']['best_val_accuracy'] for r in all_results]
    best_val_precisions = [r['validation_metrics'].get('best_val_macro_precision', 0) for r in all_results]
    best_val_recalls = [r['validation_metrics'].get('best_val_macro_recall', 0) for r in all_results]
    
    # Create statistics table for overall metrics
    overall_stats = [
        calculate_stats(test_f1s, "Test Macro F1"),
        calculate_stats(test_accs, "Test Accuracy"),
        calculate_stats(test_precisions, "Test Macro Precision"),
        calculate_stats(test_recalls, "Test Macro Recall"),
        calculate_stats(best_val_f1s, "Best Val Macro F1"),
        calculate_stats(best_val_accs, "Best Val Accuracy"),
        calculate_stats(best_val_precisions, "Best Val Macro Precision"),
        calculate_stats(best_val_recalls, "Best Val Macro Recall")
    ]
    
    df_overall_stats = pd.DataFrame(overall_stats)
    
    print("\n" + "="*80)
    print("OVERALL METRICS - Statistical Summary (30 runs)")
    print("="*80)
    print(df_overall_stats.to_string(index=False))
    
    # Save overall statistics
    overall_stats_path = os.path.join(METRICS_DIR, "overall_metrics_statistics.json")
    with open(overall_stats_path, 'w') as f:
        json.dump({"overall_metrics": overall_stats}, f, indent=2)
    
    df_overall_stats.to_csv(os.path.join(METRICS_DIR, "overall_metrics_summary.csv"), index=False)
    
    print(f"\n✅ Overall statistics saved to:")
    print(f"   - {overall_stats_path}")
    print(f"   - {os.path.join(METRICS_DIR, 'overall_metrics_summary.csv')}")
else:
    print("⚠️  No results available for statistical analysis")


In [ ]:
if len(all_results) > 0:
    # Extract per-class metrics for each style
    per_class_stats = {}
    
    for style in all_styles:
        # Extract F1, Precision, Recall, Accuracy for this style across all runs
        test_f1s_style = []
        test_precisions_style = []
        test_recalls_style = []
        test_accs_style = []
        
        val_f1s_style = []
        val_precisions_style = []
        val_recalls_style = []
        val_accs_style = []
        
        for result in all_results:
            # Test metrics
            test_pc = result['test_metrics'].get('per_class_metrics', {})
            if test_pc and 'f1' in test_pc and style in test_pc['f1']:
                test_f1s_style.append(test_pc['f1'][style])
                test_precisions_style.append(test_pc['precision'][style])
                test_recalls_style.append(test_pc['recall'][style])
                test_accs_style.append(test_pc.get('accuracy', {}).get(style, 0))
            
            # Validation metrics
            val_pc = result['validation_metrics'].get('per_class_metrics', {})
            if val_pc and 'f1' in val_pc and style in val_pc['f1']:
                val_f1s_style.append(val_pc['f1'][style])
                val_precisions_style.append(val_pc['precision'][style])
                val_recalls_style.append(val_pc['recall'][style])
                val_accs_style.append(val_pc.get('accuracy', {}).get(style, 0))
        
        # Calculate statistics for this style
        per_class_stats[style] = {
            'test': {
                'f1': calculate_stats(test_f1s_style, f"{style} - Test F1") if test_f1s_style else None,
                'precision': calculate_stats(test_precisions_style, f"{style} - Test Precision") if test_precisions_style else None,
                'recall': calculate_stats(test_recalls_style, f"{style} - Test Recall") if test_recalls_style else None,
                'accuracy': calculate_stats(test_accs_style, f"{style} - Test Accuracy") if test_accs_style else None
            },
            'validation': {
                'f1': calculate_stats(val_f1s_style, f"{style} - Val F1") if val_f1s_style else None,
                'precision': calculate_stats(val_precisions_style, f"{style} - Val Precision") if val_precisions_style else None,
                'recall': calculate_stats(val_recalls_style, f"{style} - Val Recall") if val_recalls_style else None,
                'accuracy': calculate_stats(val_accs_style, f"{style} - Val Accuracy") if val_accs_style else None
            }
        }
    
    # Create per-class summary tables (Test F1, Precision, Recall, Accuracy)
    per_class_data_f1 = []
    per_class_data_precision = []
    per_class_data_recall = []
    per_class_data_accuracy = []
    
    for style in all_styles:
        if per_class_stats[style]['test']['f1']:
            # F1 summary
            stats_f1 = per_class_stats[style]['test']['f1']
            per_class_data_f1.append({
                'Style': style,
                'Mean': stats_f1['mean'],
                'Std': stats_f1['std'],
                'Min': stats_f1['min'],
                'Max': stats_f1['max'],
                'CI_95_Lower': stats_f1['ci_95_lower'],
                'CI_95_Upper': stats_f1['ci_95_upper'],
                'CV_%': stats_f1['cv_percent']
            })
            
            # Precision summary
            stats_prec = per_class_stats[style]['test']['precision']
            per_class_data_precision.append({
                'Style': style,
                'Mean': stats_prec['mean'],
                'Std': stats_prec['std'],
                'Min': stats_prec['min'],
                'Max': stats_prec['max'],
                'CI_95_Lower': stats_prec['ci_95_lower'],
                'CI_95_Upper': stats_prec['ci_95_upper'],
                'CV_%': stats_prec['cv_percent']
            })
            
            # Recall summary
            stats_rec = per_class_stats[style]['test']['recall']
            per_class_data_recall.append({
                'Style': style,
                'Mean': stats_rec['mean'],
                'Std': stats_rec['std'],
                'Min': stats_rec['min'],
                'Max': stats_rec['max'],
                'CI_95_Lower': stats_rec['ci_95_lower'],
                'CI_95_Upper': stats_rec['ci_95_upper'],
                'CV_%': stats_rec['cv_percent']
            })
            
            # Accuracy summary
            stats_acc = per_class_stats[style]['test']['accuracy']
            per_class_data_accuracy.append({
                'Style': style,
                'Mean': stats_acc['mean'],
                'Std': stats_acc['std'],
                'Min': stats_acc['min'],
                'Max': stats_acc['max'],
                'CI_95_Lower': stats_acc['ci_95_lower'],
                'CI_95_Upper': stats_acc['ci_95_upper'],
                'CV_%': stats_acc['cv_percent']
            })
    
    df_per_class_f1 = pd.DataFrame(per_class_data_f1)
    df_per_class_precision = pd.DataFrame(per_class_data_precision)
    df_per_class_recall = pd.DataFrame(per_class_data_recall)
    df_per_class_accuracy = pd.DataFrame(per_class_data_accuracy)
    
    print("\n" + "="*80)
    print("PER-CLASS METRICS - Test F1-Score (30 runs)")
    print("="*80)
    print(df_per_class_f1.to_string(index=False))
    
    print("\n" + "="*80)
    print("PER-CLASS METRICS - Test Precision (30 runs)")
    print("="*80)
    print(df_per_class_precision.to_string(index=False))
    
    print("\n" + "="*80)
    print("PER-CLASS METRICS - Test Recall (30 runs)")
    print("="*80)
    print(df_per_class_recall.to_string(index=False))
    
    print("\n" + "="*80)
    print("PER-CLASS METRICS - Test Accuracy (30 runs)")
    print("="*80)
    print(df_per_class_accuracy.to_string(index=False))
    
    # Save per-class statistics
    per_class_stats_path = os.path.join(METRICS_DIR, "per_class_metrics_statistics.json")
    with open(per_class_stats_path, 'w') as f:
        json.dump(per_class_stats, f, indent=2)
    
    df_per_class_f1.to_csv(os.path.join(METRICS_DIR, "per_class_metrics_f1_summary.csv"), index=False)
    df_per_class_precision.to_csv(os.path.join(METRICS_DIR, "per_class_metrics_precision_summary.csv"), index=False)
    df_per_class_recall.to_csv(os.path.join(METRICS_DIR, "per_class_metrics_recall_summary.csv"), index=False)
    df_per_class_accuracy.to_csv(os.path.join(METRICS_DIR, "per_class_metrics_accuracy_summary.csv"), index=False)
    
    print(f"\n✅ Per-class statistics saved to:")
    print(f"   - {per_class_stats_path}")
    print(f"   - per_class_metrics_*_summary.csv (F1, Precision, Recall, Accuracy)")
else:
    print("⚠️  No results available for per-class analysis")


In [ ]:
if len(all_results) > 0:
    # Create formatted summary table for overall metrics
    print("\n" + "="*80)
    print("COMPREHENSIVE REPORT: Overall Performance (30 runs)")
    print("="*80)
    
    # Format: Metric | Mean ± Std | 95% CI | Min | Max | CV%
    report_data = []
    for stats in overall_stats:
        report_data.append({
            'Metric': stats['metric'],
            'Mean ± Std': f"{stats['mean']:.4f} ± {stats['std']:.4f}",
            '95% CI': f"[{stats['ci_95_lower']:.4f}, {stats['ci_95_upper']:.4f}]",
            'Min': f"{stats['min']:.4f}",
            'Max': f"{stats['max']:.4f}",
            'CV %': f"{stats['cv_percent']:.2f}%"
        })
    
    df_report = pd.DataFrame(report_data)
    print(df_report.to_string(index=False))
    
    # Create formatted per-class table
    print("\n" + "="*80)
    print("COMPREHENSIVE REPORT: Per-Class Performance (30 runs)")
    print("="*80)
    
    # Format: Style | F1 (Mean ± Std) | Precision (Mean ± Std) | Recall (Mean ± Std) | Accuracy (Mean ± Std)
    per_class_report_data = []
    for style in all_styles:
        if per_class_stats[style]['test']['f1']:
            f1_stats = per_class_stats[style]['test']['f1']
            prec_stats = per_class_stats[style]['test']['precision']
            rec_stats = per_class_stats[style]['test']['recall']
            acc_stats = per_class_stats[style]['test']['accuracy']
            
            per_class_report_data.append({
                'Style': style,
                'F1-Score': f"{f1_stats['mean']:.4f} ± {f1_stats['std']:.4f}",
                'Precision': f"{prec_stats['mean']:.4f} ± {prec_stats['std']:.4f}",
                'Recall': f"{rec_stats['mean']:.4f} ± {rec_stats['std']:.4f}",
                'Accuracy': f"{acc_stats['mean']:.4f} ± {acc_stats['std']:.4f}",
                '95% CI (F1)': f"[{f1_stats['ci_95_lower']:.4f}, {f1_stats['ci_95_upper']:.4f}]"
            })
    
    df_per_class_report = pd.DataFrame(per_class_report_data)
    print(df_per_class_report.to_string(index=False))
    
    # Save comprehensive reports
    df_report.to_csv(os.path.join(METRICS_DIR, "comprehensive_report_overall.csv"), index=False)
    df_per_class_report.to_csv(os.path.join(METRICS_DIR, "comprehensive_report_per_class.csv"), index=False)
    
    print(f"\n✅ Comprehensive reports saved to:")
    print(f"   - {os.path.join(METRICS_DIR, 'comprehensive_report_overall.csv')}")
    print(f"   - {os.path.join(METRICS_DIR, 'comprehensive_report_per_class.csv')}")
else:
    print("⚠️  No results available for report generation")


## 12. Visualizations


In [ ]:
if len(all_results) > 0 and len(per_class_stats) > 0:
    # Create output directory for visualizations
    viz_dir = os.path.join(ARTIFACTS_DIR, "per_class_visualizations")
    os.makedirs(viz_dir, exist_ok=True)
    
    # Extract styles and metrics for plotting
    styles_list = []
    f1_means = []
    f1_stds = []
    prec_means = []
    prec_stds = []
    rec_means = []
    rec_stds = []
    acc_means = []
    acc_stds = []
    
    for style in all_styles:
        if per_class_stats[style]['test']['f1']:
            styles_list.append(style)
            f1_means.append(per_class_stats[style]['test']['f1']['mean'])
            f1_stds.append(per_class_stats[style]['test']['f1']['std'])
            prec_means.append(per_class_stats[style]['test']['precision']['mean'])
            prec_stds.append(per_class_stats[style]['test']['precision']['std'])
            rec_means.append(per_class_stats[style]['test']['recall']['mean'])
            rec_stds.append(per_class_stats[style]['test']['recall']['std'])
            acc_means.append(per_class_stats[style]['test']['accuracy']['mean'])
            acc_stds.append(per_class_stats[style]['test']['accuracy']['std'])
    
    # Create comprehensive visualization
    fig, axes = plt.subplots(3, 2, figsize=(16, 18))
    
    x_pos = np.arange(len(styles_list))
    
    # Plot 1: Per-Class F1-Score Bar Chart with Error Bars
    axes[0, 0].bar(x_pos, f1_means, yerr=f1_stds, capsize=5, alpha=0.7, color='skyblue', edgecolor='black')
    axes[0, 0].set_xlabel('Style')
    axes[0, 0].set_ylabel('F1-Score (Mean ± Std)')
    axes[0, 0].set_title('Per-Class F1-Score (Test Set, 30 runs)', fontsize=12, fontweight='bold')
    axes[0, 0].set_xticks(x_pos)
    axes[0, 0].set_xticklabels(styles_list, rotation=45, ha='right')
    axes[0, 0].grid(True, alpha=0.3, axis='y')
    axes[0, 0].axhline(y=np.mean(f1_means), color='red', linestyle='--', alpha=0.7, label=f'Mean: {np.mean(f1_means):.4f}')
    axes[0, 0].legend()
    
    # Plot 2: Per-Class Precision vs Recall
    axes[0, 1].scatter(prec_means, rec_means, s=100, alpha=0.6, edgecolors='black', linewidth=1)
    for i, style in enumerate(styles_list):
        axes[0, 1].annotate(style, (prec_means[i], rec_means[i]), fontsize=8, ha='center')
    axes[0, 1].set_xlabel('Precision (Mean)')
    axes[0, 1].set_ylabel('Recall (Mean)')
    axes[0, 1].set_title('Precision vs Recall (Test Set, 30 runs)', fontsize=12, fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].plot([0, 1], [0, 1], 'k--', alpha=0.3)  # Diagonal line
    
    # Plot 3: Per-Class Precision Bar Chart
    axes[1, 0].bar(x_pos, prec_means, yerr=prec_stds, capsize=5, alpha=0.7, color='lightgreen', edgecolor='black')
    axes[1, 0].set_xlabel('Style')
    axes[1, 0].set_ylabel('Precision (Mean ± Std)')
    axes[1, 0].set_title('Per-Class Precision (Test Set, 30 runs)', fontsize=12, fontweight='bold')
    axes[1, 0].set_xticks(x_pos)
    axes[1, 0].set_xticklabels(styles_list, rotation=45, ha='right')
    axes[1, 0].grid(True, alpha=0.3, axis='y')
    axes[1, 0].axhline(y=np.mean(prec_means), color='red', linestyle='--', alpha=0.7, label=f'Mean: {np.mean(prec_means):.4f}')
    axes[1, 0].legend()
    
    # Plot 4: Per-Class Recall Bar Chart
    axes[1, 1].bar(x_pos, rec_means, yerr=rec_stds, capsize=5, alpha=0.7, color='lightcoral', edgecolor='black')
    axes[1, 1].set_xlabel('Style')
    axes[1, 1].set_ylabel('Recall (Mean ± Std)')
    axes[1, 1].set_title('Per-Class Recall (Test Set, 30 runs)', fontsize=12, fontweight='bold')
    axes[1, 1].set_xticks(x_pos)
    axes[1, 1].set_xticklabels(styles_list, rotation=45, ha='right')
    axes[1, 1].grid(True, alpha=0.3, axis='y')
    axes[1, 1].axhline(y=np.mean(rec_means), color='red', linestyle='--', alpha=0.7, label=f'Mean: {np.mean(rec_means):.4f}')
    axes[1, 1].legend()
    
    # Plot 5: Per-Class Accuracy Bar Chart
    axes[2, 0].bar(x_pos, acc_means, yerr=acc_stds, capsize=5, alpha=0.7, color='lightyellow', edgecolor='black')
    axes[2, 0].set_xlabel('Style')
    axes[2, 0].set_ylabel('Accuracy (Mean ± Std)')
    axes[2, 0].set_title('Per-Class Accuracy (Test Set, 30 runs)', fontsize=12, fontweight='bold')
    axes[2, 0].set_xticks(x_pos)
    axes[2, 0].set_xticklabels(styles_list, rotation=45, ha='right')
    axes[2, 0].grid(True, alpha=0.3, axis='y')
    axes[2, 0].axhline(y=np.mean(acc_means), color='red', linestyle='--', alpha=0.7, label=f'Mean: {np.mean(acc_means):.4f}')
    axes[2, 0].legend()
    
    # Plot 6: Heatmap of Per-Class Metrics
    metrics_matrix = []
    for style in styles_list:
        metrics_matrix.append([
            per_class_stats[style]['test']['f1']['mean'],
            per_class_stats[style]['test']['precision']['mean'],
            per_class_stats[style]['test']['recall']['mean'],
            per_class_stats[style]['test']['accuracy']['mean']
        ])
    
    metrics_matrix = np.array(metrics_matrix)
    sns.heatmap(metrics_matrix, 
                xticklabels=['F1', 'Precision', 'Recall', 'Accuracy'],
                yticklabels=styles_list,
                annot=True, fmt='.3f', cmap='YlOrRd',
                cbar_kws={'label': 'Score'}, ax=axes[2, 1])
    axes[2, 1].set_title('Per-Class Metrics Heatmap (Test Set, 30 runs)', fontsize=12, fontweight='bold')
    axes[2, 1].set_ylabel('Style')
    
    plt.suptitle('Text-Only Model: Per-Class Performance Analysis (30 Robustness Experiments)', 
                 fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    # Save plot
    plot_path = os.path.join(viz_dir, "per_class_metrics_analysis.png")
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✅ Per-class visualizations saved to: {plot_path}")
    
    # Additional plots: Box plots for F1, Precision, Recall, Accuracy distributions
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Extract values for each class across all runs
    f1_data_by_class = []
    prec_data_by_class = []
    rec_data_by_class = []
    acc_data_by_class = []
    
    for style in styles_list:
        f1_values = []
        prec_values = []
        rec_values = []
        acc_values = []
        for result in all_results:
            test_pc = result['test_metrics'].get('per_class_metrics', {})
            if test_pc and 'f1' in test_pc and style in test_pc['f1']:
                f1_values.append(test_pc['f1'][style])
                prec_values.append(test_pc['precision'][style])
                rec_values.append(test_pc['recall'][style])
                acc_values.append(test_pc.get('accuracy', {}).get(style, 0))
        f1_data_by_class.append(f1_values)
        prec_data_by_class.append(prec_values)
        rec_data_by_class.append(rec_values)
        acc_data_by_class.append(acc_values)
    
    # F1 distribution
    bp = axes[0, 0].boxplot(f1_data_by_class, labels=styles_list, patch_artist=True,
                    boxprops=dict(facecolor='lightblue', alpha=0.7),
                    medianprops=dict(color='red', linewidth=2))
    axes[0, 0].set_xlabel('Style', fontsize=12)
    axes[0, 0].set_ylabel('F1-Score', fontsize=12)
    axes[0, 0].set_title('Per-Class F1-Score Distribution (30 runs)', fontsize=12, fontweight='bold')
    axes[0, 0].tick_params(axis='x', rotation=45)
    axes[0, 0].grid(True, alpha=0.3, axis='y')
    
    # Precision distribution
    bp = axes[0, 1].boxplot(prec_data_by_class, labels=styles_list, patch_artist=True,
                    boxprops=dict(facecolor='lightgreen', alpha=0.7),
                    medianprops=dict(color='red', linewidth=2))
    axes[0, 1].set_xlabel('Style', fontsize=12)
    axes[0, 1].set_ylabel('Precision', fontsize=12)
    axes[0, 1].set_title('Per-Class Precision Distribution (30 runs)', fontsize=12, fontweight='bold')
    axes[0, 1].tick_params(axis='x', rotation=45)
    axes[0, 1].grid(True, alpha=0.3, axis='y')
    
    # Recall distribution
    bp = axes[1, 0].boxplot(rec_data_by_class, labels=styles_list, patch_artist=True,
                    boxprops=dict(facecolor='lightcoral', alpha=0.7),
                    medianprops=dict(color='red', linewidth=2))
    axes[1, 0].set_xlabel('Style', fontsize=12)
    axes[1, 0].set_ylabel('Recall', fontsize=12)
    axes[1, 0].set_title('Per-Class Recall Distribution (30 runs)', fontsize=12, fontweight='bold')
    axes[1, 0].tick_params(axis='x', rotation=45)
    axes[1, 0].grid(True, alpha=0.3, axis='y')
    
    # Accuracy distribution
    bp = axes[1, 1].boxplot(acc_data_by_class, labels=styles_list, patch_artist=True,
                    boxprops=dict(facecolor='lightyellow', alpha=0.7),
                    medianprops=dict(color='red', linewidth=2))
    axes[1, 1].set_xlabel('Style', fontsize=12)
    axes[1, 1].set_ylabel('Accuracy', fontsize=12)
    axes[1, 1].set_title('Per-Class Accuracy Distribution (30 runs)', fontsize=12, fontweight='bold')
    axes[1, 1].tick_params(axis='x', rotation=45)
    axes[1, 1].grid(True, alpha=0.3, axis='y')
    
    plt.suptitle('Text-Only Model: Per-Class Metrics Distribution (30 runs)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    boxplot_path = os.path.join(viz_dir, "per_class_metrics_distribution.png")
    plt.savefig(boxplot_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✅ Metrics distribution box plots saved to: {boxplot_path}")
    
    # Overall metrics distribution plots
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Test F1 distribution
    axes[0, 0].boxplot([test_f1s], vert=True, patch_artist=True,
                       boxprops=dict(facecolor='lightblue', alpha=0.7),
                       medianprops=dict(color='red', linewidth=2))
    axes[0, 0].scatter([1] * len(test_f1s), test_f1s, alpha=0.3, s=20, color='gray')
    axes[0, 0].axhline(y=np.mean(test_f1s), color='green', linestyle='--', 
                      label=f'Mean: {np.mean(test_f1s):.4f}')
    axes[0, 0].set_ylabel('Test Macro F1-Score')
    axes[0, 0].set_title('Test Macro F1 Distribution', fontsize=12, fontweight='bold')
    axes[0, 0].grid(True, alpha=0.3, axis='y')
    axes[0, 0].legend()
    
    # Test Accuracy distribution
    axes[0, 1].boxplot([test_accs], vert=True, patch_artist=True,
                       boxprops=dict(facecolor='lightcoral', alpha=0.7),
                       medianprops=dict(color='red', linewidth=2))
    axes[0, 1].scatter([1] * len(test_accs), test_accs, alpha=0.3, s=20, color='gray')
    axes[0, 1].axhline(y=np.mean(test_accs), color='green', linestyle='--', 
                      label=f'Mean: {np.mean(test_accs):.2f}%')
    axes[0, 1].set_ylabel('Test Accuracy (%)')
    axes[0, 1].set_title('Test Accuracy Distribution', fontsize=12, fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3, axis='y')
    axes[0, 1].legend()
    
    # Test Precision distribution
    axes[1, 0].boxplot([test_precisions], vert=True, patch_artist=True,
                       boxprops=dict(facecolor='lightgreen', alpha=0.7),
                       medianprops=dict(color='red', linewidth=2))
    axes[1, 0].scatter([1] * len(test_precisions), test_precisions, alpha=0.3, s=20, color='gray')
    axes[1, 0].axhline(y=np.mean(test_precisions), color='green', linestyle='--', 
                      label=f'Mean: {np.mean(test_precisions):.4f}')
    axes[1, 0].set_ylabel('Test Macro Precision')
    axes[1, 0].set_title('Test Macro Precision Distribution', fontsize=12, fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3, axis='y')
    axes[1, 0].legend()
    
    # Test Recall distribution
    axes[1, 1].boxplot([test_recalls], vert=True, patch_artist=True,
                       boxprops=dict(facecolor='lightyellow', alpha=0.7),
                       medianprops=dict(color='red', linewidth=2))
    axes[1, 1].scatter([1] * len(test_recalls), test_recalls, alpha=0.3, s=20, color='gray')
    axes[1, 1].axhline(y=np.mean(test_recalls), color='green', linestyle='--', 
                      label=f'Mean: {np.mean(test_recalls):.4f}')
    axes[1, 1].set_ylabel('Test Macro Recall')
    axes[1, 1].set_title('Test Macro Recall Distribution', fontsize=12, fontweight='bold')
    axes[1, 1].grid(True, alpha=0.3, axis='y')
    axes[1, 1].legend()
    
    plt.suptitle('Text-Only Model: Overall Metrics Distribution (30 runs)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    overall_dist_path = os.path.join(viz_dir, "overall_metrics_distribution.png")
    plt.savefig(overall_dist_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"✅ Overall metrics distribution plots saved to: {overall_dist_path}")
else:
    print("⚠️  No results available for visualization")


In [ ]:
if len(all_results) > 0:
    print("\n" + "="*80)
    print("FINAL SUMMARY - Text-Only Model Robustness Experiments")
    print("="*80)
    
    # Overall performance
    test_f1_mean = np.mean([r['test_metrics']['test_macro_f1'] for r in all_results])
    test_f1_std = np.std([r['test_metrics']['test_macro_f1'] for r in all_results])
    test_acc_mean = np.mean([r['test_metrics']['test_accuracy'] for r in all_results])
    test_acc_std = np.std([r['test_metrics']['test_accuracy'] for r in all_results])
    test_prec_mean = np.mean([r['test_metrics'].get('test_macro_precision', 0) for r in all_results])
    test_prec_std = np.std([r['test_metrics'].get('test_macro_precision', 0) for r in all_results])
    test_rec_mean = np.mean([r['test_metrics'].get('test_macro_recall', 0) for r in all_results])
    test_rec_std = np.std([r['test_metrics'].get('test_macro_recall', 0) for r in all_results])
    
    print(f"\n📊 PRIMARY METRIC (Test Macro F1):")
    print(f"   {test_f1_mean:.4f} ± {test_f1_std:.4f}")
    
    print(f"\n📈 Overall Test Performance:")
    print(f"   Accuracy: {test_acc_mean:.2f}% ± {test_acc_std:.2f}%")
    print(f"   Precision: {test_prec_mean:.4f} ± {test_prec_std:.4f}")
    print(f"   Recall: {test_rec_mean:.4f} ± {test_rec_std:.4f}")
    
    # Per-class summary
    if len(per_class_stats) > 0:
        print(f"\n📋 Per-Class Performance Summary:")
        print(f"   Best F1 class: {max(styles_list, key=lambda s: per_class_stats[s]['test']['f1']['mean'])}")
        print(f"   Worst F1 class: {min(styles_list, key=lambda s: per_class_stats[s]['test']['f1']['mean'])}")
        print(f"   Most stable class (lowest CV): {min(styles_list, key=lambda s: per_class_stats[s]['test']['f1']['cv_percent'])}")
        print(f"   Least stable class (highest CV): {max(styles_list, key=lambda s: per_class_stats[s]['test']['f1']['cv_percent'])}")
    
    # Overfitting summary
    overfitting_flags = [r['overfitting_analysis']['overfitting_detected'] for r in all_results]
    print(f"\n🔍 Overfitting Analysis:")
    print(f"   Overfitting detected: {sum(overfitting_flags)}/{len(overfitting_flags)} experiments ({sum(overfitting_flags)/len(overfitting_flags)*100:.1f}%)")
    
    # Training time
    training_times = [r['training_info']['total_time_minutes'] for r in all_results]
    print(f"\n⏱️  Training Time:")
    print(f"   Average: {np.mean(training_times):.1f} minutes per experiment")
    print(f"   Total: {np.sum(training_times):.1f} minutes ({np.sum(training_times)/60:.1f} hours)")
    
    print(f"\n✅ Outputs under: {EXPERIMENT_ROOT}")
    print(f"   - Overall metrics: overall_metrics_summary.csv")
    print(f"   - Per-class metrics: per_class_metrics_*_summary.csv (F1, Precision, Recall, Accuracy)")
    print(f"   - Comprehensive reports: comprehensive_report_*.csv")
    print(f"   - Visualizations: per_class_visualizations/")
    
    print("\n" + "="*80)
else:
    print("⚠️  No results available for final summary")
